[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1YrfMl6sRIl-vVptTF14QM708E7q3V4zR/view?usp=drive_link)

# RAG Evaluation – Full Dataset

This notebook demonstrates how to evaluate RAG pipelines when you have both model responses and the retrieved contexts. Floeval scores answer relevancy and faithfulness to the retrieved documents.

**Objectives**
- Install Floeval and configure credentials
- Load a RAG dataset from a JSON file you provide
- Run `answer_relevancy` and `faithfulness` metrics
- Inspect aggregate and per-sample results

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
%pip install floeval>=0.2.0b1

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass
# LLM and API configuration

OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("your-api-key")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

Import evaluation components and the LLM configuration schema.

In [ ]:
from pathlib import Path

from floeval import Evaluation, DatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 4. Configure the LLM

The LLM configuration is built using the constants defined above. RAG metrics require both the chat model and the embedding model.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

## 5. Load the RAG Dataset

Minimal JSON shape:

```json
{
  "samples": [
    {
      "user_input": "...",
      "llm_response": "...",
      "contexts": ["retrieved chunk 1", "..."],
      "ground_truth": "..."
    }
  ]
}
```

**Example file**  
<a href="../datasets/rag_evaluation/sample_rag_full_dataset.json" download="sample_rag_full_dataset.json">sample_rag_full_dataset.json</a>

Provide the dataset JSON path (or upload in Colab), then load it with `DatasetLoader`.


In [ ]:
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload your dataset JSON file:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    dataset_path = Path(next(iter(uploaded.keys())))
else:
    dataset_path = Path(input("Enter path to dataset JSON file: ").strip().strip('"')).expanduser()


### Resolve Dataset Path

Provide `dataset_path` via file upload in Colab or local JSON path input in Jupyter.


In [ ]:
dataset = DatasetLoader.from_file(dataset_path, partial_dataset=False)
print(f"Dataset loaded from {dataset_path}: {len(dataset.samples)} samples")


### Load Full RAG Dataset

Load and validate the RAG dataset with `DatasetLoader.from_file(..., partial_dataset=False)`.


## 6. Create and Run the Evaluation

Configure `Evaluation` with `answer_relevancy` and `faithfulness`, then run scoring.

In [ ]:
evaluation = Evaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["answer_relevancy", "faithfulness"],
    default_provider="ragas",
)


### Build Evaluation Object

Configure `Evaluation` with RAG metrics and provider configuration.


In [ ]:
results = evaluation.run()
print("Aggregate scores:", results.aggregate_scores)


### Run Evaluation

Execute `evaluation.run()` to compute and print aggregate metric scores.


## 7. Inspect Per-Sample Results

Each sample includes scores for both metrics. Faithfulness indicates how well the answer stays grounded in the retrieved context.

### Inspect per-sample results

Iterates `results.sample_results` to print each question snippet and metric scores.


In [ ]:
for i, sr in enumerate(results.sample_results, start=1):
    print(f"Sample {i}: {sr['user_input'][:50]}...")
    for key, data in sr.get("metrics", {}).items():
        print(f"  {key}: score={data.get('score')}")

---

## Summary

This notebook demonstrated how to evaluate RAG pipelines when you already have model responses and retrieved contexts.

The key components included:

1. **Dataset Loading**: A RAG dataset was loaded from JSON with `user_input`, `llm_response`, `contexts`, and optional `ground_truth`.
2. **Metrics**: `answer_relevancy` and `faithfulness` were run via the RAGAS provider.
3. **Results**: Aggregate scores and per-sample metrics were inspected.
4. **RAG notes**: `contexts` support faithfulness and retrieval-style metrics; `ground_truth` enables context_precision and context_recall; RAGAS and DeepEval metrics can be combined in one run.

This example showcases evaluating pre-generated RAG outputs with Floeval.